# Phase 1 — Combined Binary Classification Baselines
**Runtime:** Google Colab — **GPU (T4 or A100)**

**Purpose:** Unified notebook combining NB02 (XGBoost), NB03 (MOSAIKS Ridge),
NB04 (ResNet-18 LogReg), NB05 (DINOv2 LogReg), and NB07 (Prithvi linear probe)
into a single run. Data is loaded **once** and shared across all five models.

| # | Model | Features | Classifier |
|---|---|---|---|
| 1 | XGBoost Engineered | 35-d hand-crafted | XGBClassifier |
| 2 | MOSAIKS + Ridge | 4096-d random conv | RidgeClassifierCV |
| 3 | ResNet-18 + LogReg | 512-d frozen backbone | LogisticRegressionCV |
| 4 | DINOv2 + LogReg | 768-d frozen CLS token | LogisticRegressionCV |
| 5 | Prithvi-300M Linear | 1024-d frozen encoder | LayerNorm→Dropout→Linear (Lightning) |

**Task:** Binary classification — `has_coverage` from `label_confidence == 'confirmed_positive'`

**Evaluation:** Val-optimal threshold via PR-curve F1 maximisation, then test-set metrics:
PR-AUC, ROC-AUC, F1, Precision, Recall, Accuracy, Confusion Matrix, PR Curve.

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/sampled_patches.csv` — patch metadata + labels

## Step 0: Environment Setup

In [ ]:
# ============================================================
# Cell 0.1: Install dependencies (includes terratorch for Prithvi)
# Pin numpy BEFORE terratorch so it cannot be upgraded to 2.x
# ============================================================
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q xgboost rasterio scikit-learn matplotlib seaborn

# Restart runtime so all C extensions load against the pinned numpy
import os
print("Restarting runtime to apply numpy pin — re-run from Cell 0.2 after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# Cell 0.2: Clone repo
# ============================================================
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# ============================================================
# Cell 0.3: Sync patches from Google Drive
# ============================================================
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Local patches: {local_count}')

if local_count < 6900:
    print('Syncing from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
    print(f'Patches after sync: {local_count}')
else:
    print('Patches already available locally.')

## Step 1: Imports & Constants

In [ ]:
i
m
p
o
r
t
 
o
s

i
m
p
o
r
t
 
r
a
n
d
o
m

i
m
p
o
r
t
 
n
u
m
p
y
 
a
s
 
n
p

i
m
p
o
r
t
 
p
a
n
d
a
s
 
a
s
 
p
d

i
m
p
o
r
t
 
r
a
s
t
e
r
i
o

i
m
p
o
r
t
 
t
o
r
c
h

i
m
p
o
r
t
 
t
o
r
c
h
.
n
n
 
a
s
 
n
n

i
m
p
o
r
t
 
t
o
r
c
h
.
n
n
.
f
u
n
c
t
i
o
n
a
l
 
a
s
 
F
u
n
c

i
m
p
o
r
t
 
t
o
r
c
h
v
i
s
i
o
n
.
m
o
d
e
l
s
 
a
s
 
m
o
d
e
l
s

i
m
p
o
r
t
 
p
y
t
o
r
c
h
_
l
i
g
h
t
n
i
n
g
 
a
s
 
p
l

i
m
p
o
r
t
 
x
g
b
o
o
s
t
 
a
s
 
x
g
b

i
m
p
o
r
t
 
m
a
t
p
l
o
t
l
i
b
.
p
y
p
l
o
t
 
a
s
 
p
l
t

i
m
p
o
r
t
 
s
e
a
b
o
r
n
 
a
s
 
s
n
s

i
m
p
o
r
t
 
w
a
r
n
i
n
g
s

i
m
p
o
r
t
 
l
o
g
g
i
n
g

i
m
p
o
r
t
 
s
u
b
p
r
o
c
e
s
s

f
r
o
m
 
p
a
t
h
l
i
b
 
i
m
p
o
r
t
 
P
a
t
h

f
r
o
m
 
t
o
r
c
h
.
u
t
i
l
s
.
d
a
t
a
 
i
m
p
o
r
t
 
D
a
t
a
s
e
t
,
 
D
a
t
a
L
o
a
d
e
r
,
 
W
e
i
g
h
t
e
d
R
a
n
d
o
m
S
a
m
p
l
e
r

f
r
o
m
 
s
k
l
e
a
r
n
.
l
i
n
e
a
r
_
m
o
d
e
l
 
i
m
p
o
r
t
 
R
i
d
g
e
C
l
a
s
s
i
f
i
e
r
C
V
,
 
L
o
g
i
s
t
i
c
R
e
g
r
e
s
s
i
o
n
C
V

f
r
o
m
 
s
k
l
e
a
r
n
.
p
r
e
p
r
o
c
e
s
s
i
n
g
 
i
m
p
o
r
t
 
S
t
a
n
d
a
r
d
S
c
a
l
e
r

f
r
o
m
 
s
k
l
e
a
r
n
.
m
e
t
r
i
c
s
 
i
m
p
o
r
t
 
(

 
 
 
 
a
c
c
u
r
a
c
y
_
s
c
o
r
e
,
 
f
1
_
s
c
o
r
e
,
 
r
o
c
_
a
u
c
_
s
c
o
r
e
,

 
 
 
 
a
v
e
r
a
g
e
_
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
,
 
p
r
e
c
i
s
i
o
n
_
r
e
c
a
l
l
_
c
u
r
v
e
,

 
 
 
 
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
,
 
r
e
c
a
l
l
_
s
c
o
r
e
,
 
c
o
n
f
u
s
i
o
n
_
m
a
t
r
i
x
,

 
 
 
 
c
l
a
s
s
i
f
i
c
a
t
i
o
n
_
r
e
p
o
r
t
,

)

f
r
o
m
 
s
k
l
e
a
r
n
.
m
o
d
e
l
_
s
e
l
e
c
t
i
o
n
 
i
m
p
o
r
t
 
t
r
a
i
n
_
t
e
s
t
_
s
p
l
i
t

f
r
o
m
 
t
q
d
m
.
a
u
t
o
 
i
m
p
o
r
t
 
t
q
d
m


w
a
r
n
i
n
g
s
.
f
i
l
t
e
r
w
a
r
n
i
n
g
s
(
'
i
g
n
o
r
e
'
)

l
o
g
g
i
n
g
.
g
e
t
L
o
g
g
e
r
(
'
r
a
s
t
e
r
i
o
'
)
.
s
e
t
L
e
v
e
l
(
l
o
g
g
i
n
g
.
E
R
R
O
R
)


#
 
─
─
 
D
e
t
e
r
m
i
n
i
s
t
i
c
 
s
e
e
d
i
n
g
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

S
E
E
D
 
=
 
4
2

r
a
n
d
o
m
.
s
e
e
d
(
S
E
E
D
)

n
p
.
r
a
n
d
o
m
.
s
e
e
d
(
S
E
E
D
)

t
o
r
c
h
.
m
a
n
u
a
l
_
s
e
e
d
(
S
E
E
D
)

t
o
r
c
h
.
c
u
d
a
.
m
a
n
u
a
l
_
s
e
e
d
_
a
l
l
(
S
E
E
D
)

t
o
r
c
h
.
b
a
c
k
e
n
d
s
.
c
u
d
n
n
.
d
e
t
e
r
m
i
n
i
s
t
i
c
 
=
 
T
r
u
e

t
o
r
c
h
.
b
a
c
k
e
n
d
s
.
c
u
d
n
n
.
b
e
n
c
h
m
a
r
k
 
=
 
F
a
l
s
e

t
r
y
:

 
 
 
 
t
o
r
c
h
.
u
s
e
_
d
e
t
e
r
m
i
n
i
s
t
i
c
_
a
l
g
o
r
i
t
h
m
s
(
T
r
u
e
)

e
x
c
e
p
t
 
E
x
c
e
p
t
i
o
n
:

 
 
 
 
p
a
s
s


H
L
S
_
M
E
A
N
S
 
=
 
[
0
.
1
4
2
4
5
4
9
5
,
 
0
.
1
3
9
2
1
4
8
1
,
 
0
.
1
2
4
3
4
6
3
1
,
 
0
.
3
1
4
2
0
0
8
9
,
 
0
.
2
0
7
4
3
5
2
6
,
 
0
.
1
2
0
4
6
5
0
3
]

H
L
S
_
S
T
D
S
 
 
=
 
[
0
.
0
4
0
3
6
2
3
1
,
 
0
.
0
4
1
8
6
9
8
3
,
 
0
.
0
5
2
6
7
6
4
6
,
 
0
.
0
8
2
2
2
2
1
0
,
 
0
.
0
6
8
3
4
7
7
4
,
 
0
.
0
5
2
9
4
2
0
5
]

S
C
A
L
E
 
 
 
 
 
=
 
0
.
0
0
0
1

B
A
N
D
_
N
A
M
E
S
 
=
 
[
'
B
l
u
e
'
,
 
'
G
r
e
e
n
'
,
 
'
R
e
d
'
,
 
'
N
I
R
'
,
 
'
S
W
I
R
1
'
,
 
'
S
W
I
R
2
'
]


P
A
T
C
H
_
D
I
R
 
 
 
=
 
'
d
a
t
a
/
r
a
w
/
p
a
t
c
h
e
s
_
2
0
1
9
'

R
E
S
U
L
T
S
_
D
I
R
 
=
 
P
a
t
h
(
'
o
u
t
p
u
t
s
/
r
e
s
u
l
t
s
'
)

F
I
G
U
R
E
S
_
D
I
R
 
=
 
P
a
t
h
(
'
o
u
t
p
u
t
s
/
f
i
g
u
r
e
s
'
)

M
O
D
E
L
S
_
D
I
R
 
 
=
 
P
a
t
h
(
'
o
u
t
p
u
t
s
/
m
o
d
e
l
s
'
)

f
o
r
 
p
 
i
n
 
[
R
E
S
U
L
T
S
_
D
I
R
,
 
F
I
G
U
R
E
S
_
D
I
R
,
 
M
O
D
E
L
S
_
D
I
R
]
:

 
 
 
 
p
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)

P
a
t
h
(
'
c
h
e
c
k
p
o
i
n
t
s
'
)
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)


D
E
V
I
C
E
 
=
 
'
c
u
d
a
'
 
i
f
 
t
o
r
c
h
.
c
u
d
a
.
i
s
_
a
v
a
i
l
a
b
l
e
(
)
 
e
l
s
e
 
'
c
p
u
'

p
r
i
n
t
(
f
'
U
s
i
n
g
 
d
e
v
i
c
e
:
 
{
D
E
V
I
C
E
}
'
)

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels,
aggregate targets, stratification features, and geographic split.

In [ ]:
df = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
df = df[df['patch_id'].isin(available)].reset_index(drop=True)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

y_train = train_df['has_coverage'].values
y_val   = val_df['has_coverage'].values
y_test  = test_df['has_coverage'].values

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Train positive: {train_df["has_coverage"].mean():.2%}')
print(f'Val   positive: {val_df["has_coverage"].mean():.2%}')
print(f'Test  positive: {test_df["has_coverage"].mean():.2%}')

## Step 4: Dataset Classes

In [ ]:
class PatchDataset(Dataset):
    """Loads 6-band HLS patches with standard normalization."""
    def __init__(self, df, patch_dir, n_bands=6):
        self.df        = df.reset_index(drop=True)
        self.patch_dir = patch_dir
        self.n_bands   = n_bands

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f"{self.patch_dir}/{row['patch_id']}.tif"
        with rasterio.open(path) as src:
            img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
        img *= SCALE
        for b in range(self.n_bands):
            img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
        img   = np.nan_to_num(img, nan=0.0)
        label = int(row['has_coverage'])
        return torch.from_numpy(img), label


class PrithviPatchDataset(Dataset):
    """Adds temporal dimension T=1 for Prithvi: (C, H, W) -> (C, T=1, H, W)."""
    def __init__(self, df, patch_dir, n_bands=6):
        self.base = PatchDataset(df, patch_dir, n_bands)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        return img.unsqueeze(1), label


train_dataset = PatchDataset(train_df, PATCH_DIR)
val_dataset   = PatchDataset(val_df,   PATCH_DIR)
test_dataset  = PatchDataset(test_df,  PATCH_DIR)

prithvi_train_dataset = PrithviPatchDataset(train_df, PATCH_DIR)
prithvi_val_dataset   = PrithviPatchDataset(val_df,   PATCH_DIR)
prithvi_test_dataset  = PrithviPatchDataset(test_df,  PATCH_DIR)

print(f'Datasets — train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}')
img, lbl = train_dataset[0]
print(f'PatchDataset sample shape: {img.shape}')
img_p, _ = prithvi_train_dataset[0]
print(f'PrithviPatchDataset sample shape: {img_p.shape}')

## Step 5A: Engineered Features (35-d) — for XGBoost

In [ ]:
def extract_engineered_features(patch_path):
    """Hand-crafted spectral + spatial features from a single patch (35 features)."""
    with rasterio.open(patch_path) as src:
        img = src.read().astype(np.float32) * SCALE

    feats = {}
    for i, name in enumerate(BAND_NAMES):
        band  = img[i]
        valid = band[band > 0]
        if len(valid) == 0:
            valid = np.array([0.0])
        feats[f'{name}_mean'] = float(valid.mean())
        feats[f'{name}_std']  = float(valid.std())
        feats[f'{name}_p10']  = float(np.percentile(valid, 10))
        feats[f'{name}_p90']  = float(np.percentile(valid, 90))

    red, nir, swir1, green = img[2], img[3], img[4], img[1]

    ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red + 1e-8), 0)
    feats['NDVI_mean'] = float(ndvi.mean())
    feats['NDVI_std']  = float(ndvi.std())

    ndbi = np.where((swir1 + nir) > 0, (swir1 - nir) / (swir1 + nir + 1e-8), 0)
    feats['NDBI_mean'] = float(ndbi.mean())
    feats['NDBI_std']  = float(ndbi.std())

    mndwi = np.where((green + swir1) > 0, (green - swir1) / (green + swir1 + 1e-8), 0)
    feats['MNDWI_mean'] = float(mndwi.mean())

    feats['NIR_spatial_var'] = float(nir.var())
    feats['bright_frac'] = float((red > 0.15).mean())

    return feats


def build_feature_matrix(df, patch_dir, desc=''):
    rows, failed = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        path = f"{patch_dir}/{row['patch_id']}.tif"
        try:
            rows.append(extract_engineered_features(path))
        except Exception:
            failed.append(row['patch_id'])
            rows.append({})
    if failed:
        print(f'  Warning: {len(failed)} patches failed')
    return pd.DataFrame(rows)


print('Extracting engineered features ...')
X_train_eng = build_feature_matrix(train_df, PATCH_DIR, desc='Train')
X_val_eng   = build_feature_matrix(val_df,   PATCH_DIR, desc='Val')
X_test_eng  = build_feature_matrix(test_df,  PATCH_DIR, desc='Test')

STRAT_FEATS = ['ntl', 'builtup', 'elevation']
for feat in STRAT_FEATS:
    X_train_eng[feat] = train_df[feat].values
    X_val_eng[feat]   = val_df[feat].values
    X_test_eng[feat]  = test_df[feat].values

col_medians = X_train_eng.median()
X_train_eng = X_train_eng.fillna(col_medians)
X_val_eng   = X_val_eng.fillna(col_medians)
X_test_eng  = X_test_eng.fillna(col_medians)

y_train = train_df['has_coverage'].values
y_val   = val_df['has_coverage'].values
y_test  = test_df['has_coverage'].values

print(f'Engineered features: {X_train_eng.shape}')

## Step 5B: MOSAIKS Random Convolutional Features (4096-d)

In [ ]:
class MOSAIKSFeaturizer:
    """Random Convolutional Features (Rolf et al. 2021)."""
    def __init__(self, n_features=4096, patch_size=3, n_channels=6, seed=42, filter_chunk=256):
        rng     = np.random.RandomState(seed)
        filters = rng.randn(n_features, n_channels, patch_size, patch_size).astype(np.float32)
        self.filters      = torch.from_numpy(filters).to(DEVICE)
        self.bias         = 1.0
        self.filter_chunk = filter_chunk

    def featurize_batch(self, img_batch):
        img_batch = img_batch.to(DEVICE)
        all_out   = []
        with torch.no_grad():
            for i in range(0, self.filters.shape[0], self.filter_chunk):
                chunk = self.filters[i : i + self.filter_chunk]
                out   = Func.conv2d(img_batch, chunk, padding=1)
                out   = torch.relu(out + self.bias)
                out   = out.mean(dim=[2, 3])
                all_out.append(out.cpu())
        return torch.cat(all_out, dim=1).numpy()


def extract_mosaiks(dataset, mosaiks_model, batch_size=64, desc=''):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    for imgs, lbls in tqdm(loader, desc=desc):
        feats.append(mosaiks_model.featurize_batch(imgs))
        labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


mosaiks = MOSAIKSFeaturizer(n_features=4096, filter_chunk=256)
print('MOSAIKS featurizer ready — extracting ...')

M_train, _ = extract_mosaiks(train_dataset, mosaiks, desc='MOSAIKS Train')
M_val,   _ = extract_mosaiks(val_dataset,   mosaiks, desc='MOSAIKS Val')
M_test,  _ = extract_mosaiks(test_dataset,  mosaiks, desc='MOSAIKS Test')
print(f'MOSAIKS feature shape: {M_train.shape}')

## Step 5C: ResNet-18 Frozen Features (512-d)

In [ ]:
class ResNet18Featurizer(nn.Module):
    """Frozen ResNet-18 with first conv adapted for 6-band HLS input."""
    def __init__(self, n_input_bands=6):
        super().__init__()
        resnet   = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(n_input_bands, 64, kernel_size=7,
                             stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3:] = (
                old_conv.weight.mean(dim=1, keepdim=True)
                .expand(-1, n_input_bands - 3, -1, -1)
            )
        resnet.conv1  = new_conv
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.feat_dim = 512
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return features.squeeze(-1).squeeze(-1)


def extract_nn_features(dataset, featurizer, batch_size=64, desc=''):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    featurizer.eval()
    for imgs, lbls in tqdm(loader, desc=desc):
        feats.append(featurizer(imgs.to(DEVICE)).cpu().numpy())
        labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


resnet_featurizer = ResNet18Featurizer(n_input_bands=6).to(DEVICE).eval()
print(f'ResNet-18 ready — output dim: {resnet_featurizer.feat_dim}')

R_train, _ = extract_nn_features(train_dataset, resnet_featurizer, desc='ResNet Train')
R_val,   _ = extract_nn_features(val_dataset,   resnet_featurizer, desc='ResNet Val')
R_test,  _ = extract_nn_features(test_dataset,  resnet_featurizer, desc='ResNet Test')
print(f'ResNet-18 feature shape: {R_train.shape}')

del resnet_featurizer
torch.cuda.empty_cache()

## Step 5D: DINOv2 ViT-Base Frozen Features (768-d)

In [ ]:
class DINOv2Featurizer(nn.Module):
    """Frozen DINOv2 ViT-Base with RGB band selection from 6-band HLS."""
    def __init__(self, n_input_bands=6):
        super().__init__()
        self.dino     = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14',
                                       pretrained=True, verbose=False)
        self.rgb_idx  = [2, 1, 0]
        self.feat_dim = 768
        for param in self.dino.parameters():
            param.requires_grad = False

    def forward(self, x):
        x_rgb = x[:, self.rgb_idx, :, :]
        with torch.no_grad():
            return self.dino(x_rgb)


print('Loading DINOv2 ViT-Base ...')
dino_featurizer = DINOv2Featurizer(n_input_bands=6).to(DEVICE).eval()
print(f'DINOv2 ready — output dim: {dino_featurizer.feat_dim}')

D_train, _ = extract_nn_features(train_dataset, dino_featurizer, batch_size=32, desc='DINOv2 Train')
D_val,   _ = extract_nn_features(val_dataset,   dino_featurizer, batch_size=32, desc='DINOv2 Val')
D_test,  _ = extract_nn_features(test_dataset,  dino_featurizer, batch_size=32, desc='DINOv2 Test')
print(f'DINOv2 feature shape: {D_train.shape}')

del dino_featurizer
torch.cuda.empty_cache()

## Step 6A: Train & Evaluate — XGBoost (Engineered Features)

In [ ]:
# Check GPU for XGBoost
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                            capture_output=True, text=True)
    gpu_name = result.stdout.strip()
    xgb_device = 'cuda'
    print(f'GPU detected: {gpu_name} → using device="cuda"')
except Exception:
    xgb_device = 'cpu'
    print('No GPU detected → falling back to CPU')

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=30,
    device=xgb_device,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(
    X_train_eng, y_train,
    eval_set=[(X_val_eng, y_val)],
    verbose=50
)
print(f'Best iteration: {xgb_model.best_iteration}')

In [ ]:
y_prob_xgb = xgb_model.predict_proba(X_test_eng)[:, 1]

# Val-optimal threshold
prec_v, rec_v, thr_v = precision_recall_curve(y_val, xgb_model.predict_proba(X_val_eng)[:, 1])
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_XGB = float(thr_v[np.argmax(f1_v)])
y_pred_xgb = (y_prob_xgb >= OPT_THR_XGB).astype(int)

pr_auc_xgb = average_precision_score(y_test, y_prob_xgb)
auc_xgb    = roc_auc_score(y_test, y_prob_xgb)
f1_xgb     = f1_score(y_test, y_pred_xgb)
prec_xgb   = precision_score(y_test, y_pred_xgb, zero_division=0)
rec_xgb    = recall_score(y_test, y_pred_xgb)
acc_xgb    = accuracy_score(y_test, y_pred_xgb)

print('=== XGBoost (Engineered Features) ===')
print(f'PR-AUC  : {pr_auc_xgb:.4f}')
print(f'ROC-AUC : {auc_xgb:.4f}')
print(f'F1 @ thr={OPT_THR_XGB:.3f} : {f1_xgb:.4f}')
print(f'Precision: {prec_xgb:.4f}  Recall: {rec_xgb:.4f}  Accuracy: {acc_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['No Coverage', 'Has Coverage']))

# Diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_xgb)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_xgb)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_xgb:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_xgb], [prec_xgb], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_XGB:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('XGBoost (Engineered Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_xgboost_engineered_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost learning curve
results_xgb = xgb_model.evals_result()
val_auc_curve = results_xgb['validation_0']['auc']

plt.figure(figsize=(8, 4))
plt.plot(val_auc_curve, label='Val AUC')
plt.axvline(xgb_model.best_iteration, color='red', linestyle='--',
            label=f'Best ({xgb_model.best_iteration})')
plt.xlabel('Boosting Round'); plt.ylabel('AUC')
plt.title('XGBoost — Validation AUC Learning Curve')
plt.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_xgboost_learning_curve.png', dpi=150)
plt.show()

In [ ]:
# Feature importance
feat_imp = pd.Series(
    xgb_model.feature_importances_,
    index=X_train_eng.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
feat_imp.head(20).plot(kind='barh')
plt.xlabel('Feature Importance (gain)')
plt.title('Top 20 Features — XGBoost Engineered')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_xgboost_feature_importance.png', dpi=150)
plt.show()

## Step 6B: Train & Evaluate — MOSAIKS + RidgeClassifierCV

In [ ]:
M_trainval = np.concatenate([M_train, M_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

ridge = RidgeClassifierCV(alphas=[0.01, 0.1, 1, 10, 100])
ridge.fit(M_trainval, y_trainval)
print(f'Best alpha: {ridge.alpha_}')

# RidgeClassifierCV uses decision_function scores (not calibrated probabilities)
y_scores_val_mosaiks  = ridge.decision_function(M_val)
y_scores_test_mosaiks = ridge.decision_function(M_test)

# Val-optimal threshold
prec_v, rec_v, thr_v = precision_recall_curve(y_val, y_scores_val_mosaiks)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_MOSAIKS = float(thr_v[np.argmax(f1_v)])
y_pred_mosaiks = (y_scores_test_mosaiks >= OPT_THR_MOSAIKS).astype(int)

pr_auc_mosaiks = average_precision_score(y_test, y_scores_test_mosaiks)
auc_mosaiks    = roc_auc_score(y_test, y_scores_test_mosaiks)
f1_mosaiks     = f1_score(y_test, y_pred_mosaiks)
prec_mosaiks   = precision_score(y_test, y_pred_mosaiks, zero_division=0)
rec_mosaiks    = recall_score(y_test, y_pred_mosaiks)
acc_mosaiks    = accuracy_score(y_test, y_pred_mosaiks)

print('=== MOSAIKS + Ridge ===')
print(f'PR-AUC  : {pr_auc_mosaiks:.4f}')
print(f'ROC-AUC : {auc_mosaiks:.4f}')
print(f'F1 @ thr={OPT_THR_MOSAIKS:.3f} : {f1_mosaiks:.4f}')
print(f'Precision: {prec_mosaiks:.4f}  Recall: {rec_mosaiks:.4f}  Accuracy: {acc_mosaiks:.4f}')
print()
print(classification_report(y_test, y_pred_mosaiks, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_mosaiks)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_scores_test_mosaiks)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_mosaiks:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_mosaiks], [prec_mosaiks], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_MOSAIKS:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('MOSAIKS + Ridge', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_mosaiks_ridge_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6C: Train & Evaluate — ResNet-18 + LogisticRegressionCV

In [ ]:
R_trainval = np.concatenate([R_train, R_val], axis=0)

scaler_r       = StandardScaler()
R_trainval_sc  = scaler_r.fit_transform(R_trainval)
R_test_sc      = scaler_r.transform(R_test)
R_val_sc       = scaler_r.transform(R_val)

lr_resnet = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1, 10],
    cv=3,
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
lr_resnet.fit(R_trainval_sc, y_trainval)
print(f'Best C: {lr_resnet.C_[0]:.4f}')

y_prob_resnet     = lr_resnet.predict_proba(R_test_sc)[:, 1]
y_val_prob_resnet = lr_resnet.predict_proba(R_val_sc)[:, 1]

prec_v, rec_v, thr_v = precision_recall_curve(y_val, y_val_prob_resnet)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_RESNET = float(thr_v[np.argmax(f1_v)])
y_pred_resnet = (y_prob_resnet >= OPT_THR_RESNET).astype(int)

pr_auc_resnet = average_precision_score(y_test, y_prob_resnet)
auc_resnet    = roc_auc_score(y_test, y_prob_resnet)
f1_resnet     = f1_score(y_test, y_pred_resnet)
prec_resnet   = precision_score(y_test, y_pred_resnet, zero_division=0)
rec_resnet    = recall_score(y_test, y_pred_resnet)
acc_resnet    = accuracy_score(y_test, y_pred_resnet)

print('=== ResNet-18 + Linear Probe ===')
print(f'PR-AUC  : {pr_auc_resnet:.4f}')
print(f'ROC-AUC : {auc_resnet:.4f}')
print(f'F1 @ thr={OPT_THR_RESNET:.3f} : {f1_resnet:.4f}')
print(f'Precision: {prec_resnet:.4f}  Recall: {rec_resnet:.4f}  Accuracy: {acc_resnet:.4f}')
print()
print(classification_report(y_test, y_pred_resnet, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_resnet)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_resnet)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_resnet:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_resnet], [prec_resnet], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_RESNET:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('ResNet-18 + LogisticRegressionCV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_resnet18_linear_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6D: Train & Evaluate — DINOv2 + LogisticRegressionCV

In [ ]:
D_trainval = np.concatenate([D_train, D_val], axis=0)

scaler_d       = StandardScaler()
D_trainval_sc  = scaler_d.fit_transform(D_trainval)
D_test_sc      = scaler_d.transform(D_test)
D_val_sc       = scaler_d.transform(D_val)

lr_dino = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1, 10],
    cv=3,
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
lr_dino.fit(D_trainval_sc, y_trainval)
print(f'Best C: {lr_dino.C_[0]:.4f}')

y_prob_dino     = lr_dino.predict_proba(D_test_sc)[:, 1]
y_val_prob_dino = lr_dino.predict_proba(D_val_sc)[:, 1]

prec_v, rec_v, thr_v = precision_recall_curve(y_val, y_val_prob_dino)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_DINO = float(thr_v[np.argmax(f1_v)])
y_pred_dino = (y_prob_dino >= OPT_THR_DINO).astype(int)

pr_auc_dino = average_precision_score(y_test, y_prob_dino)
auc_dino    = roc_auc_score(y_test, y_prob_dino)
f1_dino     = f1_score(y_test, y_pred_dino)
prec_dino   = precision_score(y_test, y_pred_dino, zero_division=0)
rec_dino    = recall_score(y_test, y_pred_dino)
acc_dino    = accuracy_score(y_test, y_pred_dino)

print('=== DINOv2 ViT-Base + Linear Probe ===')
print(f'PR-AUC  : {pr_auc_dino:.4f}')
print(f'ROC-AUC : {auc_dino:.4f}')
print(f'F1 @ thr={OPT_THR_DINO:.3f} : {f1_dino:.4f}')
print(f'Precision: {prec_dino:.4f}  Recall: {rec_dino:.4f}  Accuracy: {acc_dino:.4f}')
print()
print(classification_report(y_test, y_pred_dino, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_dino)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_dino)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_dino:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_dino], [prec_dino], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_DINO:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('DINOv2 ViT-Base + LogisticRegressionCV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_dinov2_linear_eval.png', dpi=150, bbox_inches='tight')
plt.show()

#
#
 
S
t
e
p
 
6
E
:
 
T
r
a
i
n
 
&
 
E
v
a
l
u
a
t
e
 
—
 
P
r
i
t
h
v
i
-
3
0
0
M
 
L
i
n
e
a
r
 
P
r
o
b
e
 
(
C
a
c
h
e
d
 
E
m
b
e
d
d
i
n
g
s
)

L
o
a
d
 
f
r
o
z
e
n
 
1
0
2
4
-
d
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
 
f
r
o
m
 
`
o
u
t
p
u
t
s
/
f
e
a
t
u
r
e
s
/
`
 
(
e
x
t
r
a
c
t
e
d
 
b
y
 
N
B
1
6
)
.

H
e
a
d
-
o
n
l
y
 
c
l
a
s
s
i
f
i
e
r
 
t
r
a
i
n
e
d
 
o
n
 
p
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
 
—
 
n
o
 
e
n
c
o
d
e
r
 
o
n
 
G
P
U
.

In [ ]:
O
U
T
P
U
T
_
F
E
A
T
U
R
E
S
 
=
 
P
a
t
h
(
'
o
u
t
p
u
t
s
/
f
e
a
t
u
r
e
s
'
)

O
U
T
P
U
T
_
F
E
A
T
U
R
E
S
.
m
k
d
i
r
(
p
a
r
e
n
t
s
=
T
r
u
e
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)



d
e
f
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
s
p
l
i
t
_
n
a
m
e
,
 
d
f
)
:

 
 
 
 
"
"
"
L
o
a
d
 
c
a
c
h
e
d
 
f
r
o
z
e
n
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
,
 
o
r
 
e
x
t
r
a
c
t
 
+
 
c
a
c
h
e
 
t
h
e
m
.
"
"
"

 
 
 
 
p
a
t
h
 
=
 
O
U
T
P
U
T
_
F
E
A
T
U
R
E
S
 
/
 
f
'
p
r
i
t
h
v
i
_
f
r
o
z
e
n
_
e
m
b
e
d
s
_
{
s
p
l
i
t
_
n
a
m
e
}
.
n
p
z
'

 
 
 
 
i
f
 
p
a
t
h
.
e
x
i
s
t
s
(
)
:

 
 
 
 
 
 
 
 
d
a
t
a
 
=
 
n
p
.
l
o
a
d
(
p
a
t
h
)

 
 
 
 
 
 
 
 
X
 
=
 
d
a
t
a
[
'
X
'
]
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
'
 
 
L
o
a
d
e
d
 
c
a
c
h
e
d
 
{
s
p
l
i
t
_
n
a
m
e
}
:
 
{
X
.
s
h
a
p
e
}
'
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
X


 
 
 
 
p
r
i
n
t
(
f
'
 
 
C
a
c
h
e
 
m
i
s
s
 
f
o
r
 
{
s
p
l
i
t
_
n
a
m
e
}
 
—
 
e
x
t
r
a
c
t
i
n
g
 
f
r
o
m
 
s
c
r
a
t
c
h
 
.
.
.
'
)

 
 
 
 
f
r
o
m
 
t
e
r
r
a
t
o
r
c
h
.
r
e
g
i
s
t
r
y
 
i
m
p
o
r
t
 
B
A
C
K
B
O
N
E
_
R
E
G
I
S
T
R
Y


 
 
 
 
e
n
c
o
d
e
r
 
=
 
B
A
C
K
B
O
N
E
_
R
E
G
I
S
T
R
Y
.
b
u
i
l
d
(
'
p
r
i
t
h
v
i
_
e
o
_
v
2
_
3
0
0
'
,
 
p
r
e
t
r
a
i
n
e
d
=
T
r
u
e
)
.
t
o
(
D
E
V
I
C
E
)
.
e
v
a
l
(
)

 
 
 
 
f
o
r
 
p
 
i
n
 
e
n
c
o
d
e
r
.
p
a
r
a
m
e
t
e
r
s
(
)
:

 
 
 
 
 
 
 
 
p
.
r
e
q
u
i
r
e
s
_
g
r
a
d
 
=
 
F
a
l
s
e


 
 
 
 
c
l
a
s
s
 
_
P
r
i
t
h
v
i
E
x
t
r
a
c
t
D
a
t
a
s
e
t
(
D
a
t
a
s
e
t
)
:

 
 
 
 
 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
d
f
,
 
p
a
t
c
h
_
d
i
r
,
 
n
_
b
a
n
d
s
=
6
)
:

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
d
f
 
=
 
d
f
.
r
e
s
e
t
_
i
n
d
e
x
(
d
r
o
p
=
T
r
u
e
)

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
p
a
t
c
h
_
d
i
r
 
=
 
p
a
t
c
h
_
d
i
r

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
n
_
b
a
n
d
s
 
=
 
n
_
b
a
n
d
s

 
 
 
 
 
 
 
 
d
e
f
 
_
_
l
e
n
_
_
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
e
n
(
s
e
l
f
.
d
f
)

 
 
 
 
 
 
 
 
d
e
f
 
_
_
g
e
t
i
t
e
m
_
_
(
s
e
l
f
,
 
i
d
x
)
:

 
 
 
 
 
 
 
 
 
 
 
 
r
o
w
 
=
 
s
e
l
f
.
d
f
.
i
l
o
c
[
i
d
x
]

 
 
 
 
 
 
 
 
 
 
 
 
w
i
t
h
 
r
a
s
t
e
r
i
o
.
o
p
e
n
(
f
"
{
s
e
l
f
.
p
a
t
c
h
_
d
i
r
}
/
{
r
o
w
[
'
p
a
t
c
h
_
i
d
'
]
}
.
t
i
f
"
)
 
a
s
 
s
r
c
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
 
=
 
s
r
c
.
r
e
a
d
(
l
i
s
t
(
r
a
n
g
e
(
1
,
 
s
e
l
f
.
n
_
b
a
n
d
s
 
+
 
1
)
)
)
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)

 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
 
*
=
 
S
C
A
L
E

 
 
 
 
 
 
 
 
 
 
 
 
f
o
r
 
b
 
i
n
 
r
a
n
g
e
(
s
e
l
f
.
n
_
b
a
n
d
s
)
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
[
b
]
 
=
 
(
i
m
g
[
b
]
 
-
 
H
L
S
_
M
E
A
N
S
[
b
]
)
 
/
 
H
L
S
_
S
T
D
S
[
b
]

 
 
 
 
 
 
 
 
 
 
 
 
i
m
g
 
=
 
n
p
.
n
a
n
_
t
o
_
n
u
m
(
i
m
g
,
 
n
a
n
=
0
.
0
)

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
t
o
r
c
h
.
f
r
o
m
_
n
u
m
p
y
(
i
m
g
)
.
u
n
s
q
u
e
e
z
e
(
1
)
,
 
r
o
w
[
'
p
a
t
c
h
_
i
d
'
]


 
 
 
 
d
s
 
=
 
_
P
r
i
t
h
v
i
E
x
t
r
a
c
t
D
a
t
a
s
e
t
(
d
f
,
 
P
A
T
C
H
_
D
I
R
)

 
 
 
 
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(
d
s
,
 
b
a
t
c
h
_
s
i
z
e
=
6
4
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
2
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
)


 
 
 
 
a
l
l
_
e
m
b
s
 
=
 
[
]

 
 
 
 
w
i
t
h
 
t
o
r
c
h
.
n
o
_
g
r
a
d
(
)
:

 
 
 
 
 
 
 
 
f
o
r
 
x
,
 
_
 
i
n
 
t
q
d
m
(
l
o
a
d
e
r
,
 
d
e
s
c
=
f
'
E
x
t
r
a
c
t
i
n
g
 
{
s
p
l
i
t
_
n
a
m
e
}
'
)
:

 
 
 
 
 
 
 
 
 
 
 
 
x
 
=
 
x
.
t
o
(
D
E
V
I
C
E
,
 
d
t
y
p
e
=
t
o
r
c
h
.
f
l
o
a
t
3
2
)

 
 
 
 
 
 
 
 
 
 
 
 
f
e
a
t
s
 
=
 
e
n
c
o
d
e
r
(
x
)

 
 
 
 
 
 
 
 
 
 
 
 
p
o
o
l
e
d
 
=
 
f
e
a
t
s
[
-
1
]
.
m
e
a
n
(
d
i
m
=
1
)

 
 
 
 
 
 
 
 
 
 
 
 
a
l
l
_
e
m
b
s
.
a
p
p
e
n
d
(
p
o
o
l
e
d
.
c
p
u
(
)
.
n
u
m
p
y
(
)
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)
)


 
 
 
 
X
 
=
 
n
p
.
c
o
n
c
a
t
e
n
a
t
e
(
a
l
l
_
e
m
b
s
,
 
a
x
i
s
=
0
)

 
 
 
 
n
p
.
s
a
v
e
z
_
c
o
m
p
r
e
s
s
e
d
(
p
a
t
h
,
 
X
=
X
,
 
p
a
t
c
h
_
i
d
=
d
f
[
'
p
a
t
c
h
_
i
d
'
]
.
t
o
_
n
u
m
p
y
(
)
)

 
 
 
 
p
r
i
n
t
(
f
'
 
 
E
x
t
r
a
c
t
e
d
 
a
n
d
 
c
a
c
h
e
d
 
{
s
p
l
i
t
_
n
a
m
e
}
:
 
{
X
.
s
h
a
p
e
}
'
)


 
 
 
 
d
e
l
 
e
n
c
o
d
e
r

 
 
 
 
t
o
r
c
h
.
c
u
d
a
.
e
m
p
t
y
_
c
a
c
h
e
(
)

 
 
 
 
r
e
t
u
r
n
 
X



p
r
i
n
t
(
'
L
o
a
d
i
n
g
 
f
r
o
z
e
n
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
 
(
1
0
2
4
-
d
)
 
.
.
.
'
)

P
_
t
r
a
i
n
_
e
m
b
 
=
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
'
t
r
a
i
n
'
,
 
t
r
a
i
n
_
d
f
)

P
_
v
a
l
_
e
m
b
 
 
 
=
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
'
v
a
l
'
,
 
 
 
v
a
l
_
d
f
)

P
_
t
e
s
t
_
e
m
b
 
 
=
 
l
o
a
d
_
o
r
_
e
x
t
r
a
c
t
_
e
m
b
e
d
d
i
n
g
s
(
'
t
e
s
t
'
,
 
 
t
e
s
t
_
d
f
)

p
r
i
n
t
(
f
'
T
r
a
i
n
:
 
{
P
_
t
r
a
i
n
_
e
m
b
.
s
h
a
p
e
}
 
 
|
 
 
V
a
l
:
 
{
P
_
v
a
l
_
e
m
b
.
s
h
a
p
e
}
 
 
|
 
 
T
e
s
t
:
 
{
P
_
t
e
s
t
_
e
m
b
.
s
h
a
p
e
}
'
)



c
l
a
s
s
 
E
m
b
e
d
d
i
n
g
C
l
a
s
s
D
a
t
a
s
e
t
(
D
a
t
a
s
e
t
)
:

 
 
 
 
"
"
"
P
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
 
+
 
i
n
t
e
g
e
r
 
c
l
a
s
s
 
l
a
b
e
l
s
.
"
"
"

 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
e
m
b
e
d
d
i
n
g
s
,
 
l
a
b
e
l
s
)
:

 
 
 
 
 
 
 
 
s
e
l
f
.
X
 
=
 
t
o
r
c
h
.
f
r
o
m
_
n
u
m
p
y
(
e
m
b
e
d
d
i
n
g
s
)

 
 
 
 
 
 
 
 
s
e
l
f
.
y
 
=
 
t
o
r
c
h
.
t
e
n
s
o
r
(
l
a
b
e
l
s
,
 
d
t
y
p
e
=
t
o
r
c
h
.
l
o
n
g
)

 
 
 
 
d
e
f
 
_
_
l
e
n
_
_
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
e
n
(
s
e
l
f
.
X
)

 
 
 
 
d
e
f
 
_
_
g
e
t
i
t
e
m
_
_
(
s
e
l
f
,
 
i
d
x
)
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
s
e
l
f
.
X
[
i
d
x
]
,
 
s
e
l
f
.
y
[
i
d
x
]



c
l
a
s
s
 
P
r
i
t
h
v
i
C
l
a
s
s
i
f
i
e
r
(
p
l
.
L
i
g
h
t
n
i
n
g
M
o
d
u
l
e
)
:

 
 
 
 
"
"
"
H
e
a
d
-
o
n
l
y
 
b
i
n
a
r
y
 
c
l
a
s
s
i
f
i
e
r
 
o
n
 
f
r
o
z
e
n
 
P
r
i
t
h
v
i
 
e
m
b
e
d
d
i
n
g
s
 
(
1
0
2
4
-
d
)
.
"
"
"

 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
l
r
=
1
e
-
3
,
 
n
u
m
_
c
l
a
s
s
e
s
=
2
)
:

 
 
 
 
 
 
 
 
s
u
p
e
r
(
)
.
_
_
i
n
i
t
_
_
(
)

 
 
 
 
 
 
 
 
s
e
l
f
.
s
a
v
e
_
h
y
p
e
r
p
a
r
a
m
e
t
e
r
s
(
)


 
 
 
 
 
 
 
 
s
e
l
f
.
h
e
a
d
 
=
 
n
n
.
S
e
q
u
e
n
t
i
a
l
(

 
 
 
 
 
 
 
 
 
 
 
 
n
n
.
L
a
y
e
r
N
o
r
m
(
1
0
2
4
)
,

 
 
 
 
 
 
 
 
 
 
 
 
n
n
.
D
r
o
p
o
u
t
(
0
.
1
)
,

 
 
 
 
 
 
 
 
 
 
 
 
n
n
.
L
i
n
e
a
r
(
1
0
2
4
,
 
n
u
m
_
c
l
a
s
s
e
s
)
,

 
 
 
 
 
 
 
 
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
s
s
_
f
n
 
=
 
n
n
.
C
r
o
s
s
E
n
t
r
o
p
y
L
o
s
s
(
)


 
 
 
 
 
 
 
 
t
r
a
i
n
a
b
l
e
 
=
 
s
u
m
(
p
.
n
u
m
e
l
(
)
 
f
o
r
 
p
 
i
n
 
s
e
l
f
.
p
a
r
a
m
e
t
e
r
s
(
)
 
i
f
 
p
.
r
e
q
u
i
r
e
s
_
g
r
a
d
)

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
'
T
r
a
i
n
a
b
l
e
 
p
a
r
a
m
s
:
 
{
t
r
a
i
n
a
b
l
e
:
,
}
'
)


 
 
 
 
d
e
f
 
f
o
r
w
a
r
d
(
s
e
l
f
,
 
x
)
:

 
 
 
 
 
 
 
 
"
"
"
x
:
 
(
B
,
 
1
0
2
4
)
 
p
r
e
-
e
x
t
r
a
c
t
e
d
 
e
m
b
e
d
d
i
n
g
s
 
→
 
l
o
g
i
t
s
 
(
B
,
 
n
u
m
_
c
l
a
s
s
e
s
)
"
"
"

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
s
e
l
f
.
h
e
a
d
(
x
)


 
 
 
 
d
e
f
 
t
r
a
i
n
i
n
g
_
s
t
e
p
(
s
e
l
f
,
 
b
a
t
c
h
,
 
b
a
t
c
h
_
i
d
x
)
:

 
 
 
 
 
 
 
 
x
,
 
y
 
 
 
=
 
b
a
t
c
h

 
 
 
 
 
 
 
 
l
o
g
i
t
s
 
=
 
s
e
l
f
(
x
)

 
 
 
 
 
 
 
 
l
o
s
s
 
 
 
=
 
s
e
l
f
.
l
o
s
s
_
f
n
(
l
o
g
i
t
s
,
 
y
)

 
 
 
 
 
 
 
 
a
c
c
 
 
 
 
=
 
(
l
o
g
i
t
s
.
a
r
g
m
a
x
(
d
i
m
=
1
)
 
=
=
 
y
)
.
f
l
o
a
t
(
)
.
m
e
a
n
(
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
t
r
a
i
n
_
l
o
s
s
'
,
 
l
o
s
s
,
 
p
r
o
g
_
b
a
r
=
T
r
u
e
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
t
r
a
i
n
_
a
c
c
'
,
 
 
a
c
c
,
 
 
p
r
o
g
_
b
a
r
=
T
r
u
e
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
o
s
s


 
 
 
 
d
e
f
 
v
a
l
i
d
a
t
i
o
n
_
s
t
e
p
(
s
e
l
f
,
 
b
a
t
c
h
,
 
b
a
t
c
h
_
i
d
x
)
:

 
 
 
 
 
 
 
 
x
,
 
y
 
 
 
=
 
b
a
t
c
h

 
 
 
 
 
 
 
 
l
o
g
i
t
s
 
=
 
s
e
l
f
(
x
)

 
 
 
 
 
 
 
 
l
o
s
s
 
 
 
=
 
s
e
l
f
.
l
o
s
s
_
f
n
(
l
o
g
i
t
s
,
 
y
)

 
 
 
 
 
 
 
 
p
r
e
d
s
 
 
=
 
l
o
g
i
t
s
.
a
r
g
m
a
x
(
d
i
m
=
1
)

 
 
 
 
 
 
 
 
a
c
c
 
 
 
 
=
 
(
p
r
e
d
s
 
=
=
 
y
)
.
f
l
o
a
t
(
)
.
m
e
a
n
(
)

 
 
 
 
 
 
 
 
t
p
 
 
 
=
 
(
(
p
r
e
d
s
 
=
=
 
1
)
 
&
 
(
y
 
=
=
 
1
)
)
.
s
u
m
(
)
.
f
l
o
a
t
(
)

 
 
 
 
 
 
 
 
p
p
 
 
 
=
 
(
p
r
e
d
s
 
=
=
 
1
)
.
s
u
m
(
)
.
f
l
o
a
t
(
)

 
 
 
 
 
 
 
 
a
p
 
 
 
=
 
(
y
 
=
=
 
1
)
.
s
u
m
(
)
.
f
l
o
a
t
(
)

 
 
 
 
 
 
 
 
p
r
e
c
 
=
 
t
p
 
/
 
(
p
p
 
+
 
1
e
-
8
)

 
 
 
 
 
 
 
 
r
e
c
 
 
=
 
t
p
 
/
 
(
a
p
 
+
 
1
e
-
8
)

 
 
 
 
 
 
 
 
f
1
 
 
 
=
 
2
 
*
 
p
r
e
c
 
*
 
r
e
c
 
/
 
(
p
r
e
c
 
+
 
r
e
c
 
+
 
1
e
-
8
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
v
a
l
_
l
o
s
s
'
,
 
l
o
s
s
,
 
p
r
o
g
_
b
a
r
=
T
r
u
e
,
 
s
y
n
c
_
d
i
s
t
=
T
r
u
e
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
v
a
l
_
a
c
c
'
,
 
 
a
c
c
,
 
 
p
r
o
g
_
b
a
r
=
T
r
u
e
,
 
s
y
n
c
_
d
i
s
t
=
T
r
u
e
)

 
 
 
 
 
 
 
 
s
e
l
f
.
l
o
g
(
'
v
a
l
_
f
1
'
,
 
 
 
f
1
,
 
 
 
p
r
o
g
_
b
a
r
=
T
r
u
e
,
 
s
y
n
c
_
d
i
s
t
=
T
r
u
e
)


 
 
 
 
d
e
f
 
c
o
n
f
i
g
u
r
e
_
o
p
t
i
m
i
z
e
r
s
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
o
p
t
i
m
i
z
e
r
 
=
 
t
o
r
c
h
.
o
p
t
i
m
.
A
d
a
m
W
(

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
p
a
r
a
m
e
t
e
r
s
(
)
,

 
 
 
 
 
 
 
 
 
 
 
 
l
r
=
s
e
l
f
.
h
p
a
r
a
m
s
.
l
r
,

 
 
 
 
 
 
 
 
 
 
 
 
w
e
i
g
h
t
_
d
e
c
a
y
=
0
.
0
,

 
 
 
 
 
 
 
 
)

 
 
 
 
 
 
 
 
s
c
h
e
d
u
l
e
r
 
=
 
t
o
r
c
h
.
o
p
t
i
m
.
l
r
_
s
c
h
e
d
u
l
e
r
.
R
e
d
u
c
e
L
R
O
n
P
l
a
t
e
a
u
(

 
 
 
 
 
 
 
 
 
 
 
 
o
p
t
i
m
i
z
e
r
,
 
m
o
d
e
=
'
m
i
n
'
,
 
f
a
c
t
o
r
=
0
.
5
,
 
p
a
t
i
e
n
c
e
=
1
0

 
 
 
 
 
 
 
 
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
{

 
 
 
 
 
 
 
 
 
 
 
 
'
o
p
t
i
m
i
z
e
r
'
:
 
o
p
t
i
m
i
z
e
r
,

 
 
 
 
 
 
 
 
 
 
 
 
'
l
r
_
s
c
h
e
d
u
l
e
r
'
:
 
{
'
s
c
h
e
d
u
l
e
r
'
:
 
s
c
h
e
d
u
l
e
r
,
 
'
m
o
n
i
t
o
r
'
:
 
'
v
a
l
_
l
o
s
s
'
}
,

 
 
 
 
 
 
 
 
}

In [ ]:
f
r
o
m
 
p
y
t
o
r
c
h
_
l
i
g
h
t
n
i
n
g
 
i
m
p
o
r
t
 
T
r
a
i
n
e
r

f
r
o
m
 
p
y
t
o
r
c
h
_
l
i
g
h
t
n
i
n
g
.
c
a
l
l
b
a
c
k
s
 
i
m
p
o
r
t
 
E
a
r
l
y
S
t
o
p
p
i
n
g
,
 
M
o
d
e
l
C
h
e
c
k
p
o
i
n
t



P
R
I
T
H
V
I
_
B
S
 
=
 
3
2


#
 
W
e
i
g
h
t
e
d
 
s
a
m
p
l
e
r
 
o
n
 
e
m
b
e
d
d
i
n
g
 
d
a
t
a
s
e
t

l
a
b
e
l
s
_
t
r
a
i
n
 
=
 
t
r
a
i
n
_
d
f
[
'
h
a
s
_
c
o
v
e
r
a
g
e
'
]
.
v
a
l
u
e
s

c
l
a
s
s
_
c
o
u
n
t
s
 
 
=
 
n
p
.
b
i
n
c
o
u
n
t
(
l
a
b
e
l
s
_
t
r
a
i
n
)

c
l
a
s
s
_
w
e
i
g
h
t
s
 
=
 
1
.
0
 
/
 
c
l
a
s
s
_
c
o
u
n
t
s

s
a
m
p
l
e
_
w
e
i
g
h
t
s
 
=
 
c
l
a
s
s
_
w
e
i
g
h
t
s
[
l
a
b
e
l
s
_
t
r
a
i
n
]

t
r
a
i
n
_
s
a
m
p
l
e
r
 
=
 
W
e
i
g
h
t
e
d
R
a
n
d
o
m
S
a
m
p
l
e
r
(

 
 
 
 
w
e
i
g
h
t
s
=
s
a
m
p
l
e
_
w
e
i
g
h
t
s
,
 
n
u
m
_
s
a
m
p
l
e
s
=
l
e
n
(
l
a
b
e
l
s
_
t
r
a
i
n
)
,
 
r
e
p
l
a
c
e
m
e
n
t
=
T
r
u
e

)


p
r
i
t
h
v
i
_
t
r
a
i
n
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(

 
 
 
 
E
m
b
e
d
d
i
n
g
C
l
a
s
s
D
a
t
a
s
e
t
(
P
_
t
r
a
i
n
_
e
m
b
,
 
y
_
t
r
a
i
n
)
,

 
 
 
 
b
a
t
c
h
_
s
i
z
e
=
P
R
I
T
H
V
I
_
B
S
,
 
s
a
m
p
l
e
r
=
t
r
a
i
n
_
s
a
m
p
l
e
r
,

 
 
 
 
n
u
m
_
w
o
r
k
e
r
s
=
0
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
,

)

p
r
i
t
h
v
i
_
v
a
l
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(

 
 
 
 
E
m
b
e
d
d
i
n
g
C
l
a
s
s
D
a
t
a
s
e
t
(
P
_
v
a
l
_
e
m
b
,
 
y
_
v
a
l
)
,

 
 
 
 
b
a
t
c
h
_
s
i
z
e
=
P
R
I
T
H
V
I
_
B
S
 
*
 
2
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,

 
 
 
 
n
u
m
_
w
o
r
k
e
r
s
=
0
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
,

)


p
r
i
t
h
v
i
_
m
o
d
e
l
 
=
 
P
r
i
t
h
v
i
C
l
a
s
s
i
f
i
e
r
(
l
r
=
1
e
-
3
)


e
a
r
l
y
_
s
t
o
p
 
=
 
E
a
r
l
y
S
t
o
p
p
i
n
g
(

 
 
 
 
m
o
n
i
t
o
r
=
'
v
a
l
_
l
o
s
s
'
,
 
p
a
t
i
e
n
c
e
=
2
0
,
 
m
o
d
e
=
'
m
i
n
'
,
 
v
e
r
b
o
s
e
=
T
r
u
e

)

c
h
e
c
k
p
o
i
n
t
_
c
b
 
=
 
M
o
d
e
l
C
h
e
c
k
p
o
i
n
t
(

 
 
 
 
d
i
r
p
a
t
h
=
'
c
h
e
c
k
p
o
i
n
t
s
'
,

 
 
 
 
f
i
l
e
n
a
m
e
=
'
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
{
e
p
o
c
h
:
0
2
d
}
_
{
v
a
l
_
f
1
:
.
4
f
}
'
,

 
 
 
 
m
o
n
i
t
o
r
=
'
v
a
l
_
f
1
'
,
 
m
o
d
e
=
'
m
a
x
'
,
 
s
a
v
e
_
t
o
p
_
k
=
1
,
 
v
e
r
b
o
s
e
=
T
r
u
e

)


t
r
a
i
n
e
r
 
=
 
T
r
a
i
n
e
r
(

 
 
 
 
m
a
x
_
e
p
o
c
h
s
=
1
0
0
,

 
 
 
 
a
c
c
e
l
e
r
a
t
o
r
=
'
g
p
u
'
 
i
f
 
t
o
r
c
h
.
c
u
d
a
.
i
s
_
a
v
a
i
l
a
b
l
e
(
)
 
e
l
s
e
 
'
c
p
u
'
,

 
 
 
 
d
e
v
i
c
e
s
=
1
,

 
 
 
 
c
a
l
l
b
a
c
k
s
=
[
e
a
r
l
y
_
s
t
o
p
,
 
c
h
e
c
k
p
o
i
n
t
_
c
b
]
,

 
 
 
 
l
o
g
_
e
v
e
r
y
_
n
_
s
t
e
p
s
=
1
0
,

 
 
 
 
e
n
a
b
l
e
_
p
r
o
g
r
e
s
s
_
b
a
r
=
T
r
u
e
,

)


t
r
a
i
n
e
r
.
f
i
t
(
p
r
i
t
h
v
i
_
m
o
d
e
l
,
 
p
r
i
t
h
v
i
_
t
r
a
i
n
_
l
o
a
d
e
r
,
 
p
r
i
t
h
v
i
_
v
a
l
_
l
o
a
d
e
r
)

p
r
i
n
t
(
f
'
B
e
s
t
 
c
h
e
c
k
p
o
i
n
t
:
 
{
c
h
e
c
k
p
o
i
n
t
_
c
b
.
b
e
s
t
_
m
o
d
e
l
_
p
a
t
h
}
'
)

p
r
i
n
t
(
f
'
B
e
s
t
 
v
a
l
 
F
1
:
 
 
 
 
 
{
c
h
e
c
k
p
o
i
n
t
_
c
b
.
b
e
s
t
_
m
o
d
e
l
_
s
c
o
r
e
:
.
4
f
}
'
)

In [ ]:
#
 
L
o
a
d
 
b
e
s
t
 
c
h
e
c
k
p
o
i
n
t
 
a
n
d
 
e
v
a
l
u
a
t
e

b
e
s
t
_
p
r
i
t
h
v
i
 
=
 
P
r
i
t
h
v
i
C
l
a
s
s
i
f
i
e
r
.
l
o
a
d
_
f
r
o
m
_
c
h
e
c
k
p
o
i
n
t
(

 
 
 
 
c
h
e
c
k
p
o
i
n
t
_
c
b
.
b
e
s
t
_
m
o
d
e
l
_
p
a
t
h

)
.
t
o
(
D
E
V
I
C
E
)
.
e
v
a
l
(
)


#
 
V
a
l
 
i
n
f
e
r
e
n
c
e
 
→
 
o
p
t
i
m
a
l
 
t
h
r
e
s
h
o
l
d

p
r
i
t
h
v
i
_
v
a
l
_
t
e
s
t
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(

 
 
 
 
E
m
b
e
d
d
i
n
g
C
l
a
s
s
D
a
t
a
s
e
t
(
P
_
v
a
l
_
e
m
b
,
 
y
_
v
a
l
)
,

 
 
 
 
b
a
t
c
h
_
s
i
z
e
=
2
5
6
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
0
,

)

v
a
l
_
p
r
o
b
s
,
 
v
a
l
_
l
a
b
e
l
s
 
=
 
[
]
,
 
[
]

w
i
t
h
 
t
o
r
c
h
.
n
o
_
g
r
a
d
(
)
:

 
 
 
 
f
o
r
 
x
,
 
y
 
i
n
 
p
r
i
t
h
v
i
_
v
a
l
_
t
e
s
t
_
l
o
a
d
e
r
:

 
 
 
 
 
 
 
 
l
o
g
i
t
s
 
=
 
b
e
s
t
_
p
r
i
t
h
v
i
(
x
.
t
o
(
D
E
V
I
C
E
)
)

 
 
 
 
 
 
 
 
p
r
o
b
s
 
 
=
 
t
o
r
c
h
.
s
o
f
t
m
a
x
(
l
o
g
i
t
s
,
 
d
i
m
=
1
)
[
:
,
 
1
]

 
 
 
 
 
 
 
 
v
a
l
_
p
r
o
b
s
.
e
x
t
e
n
d
(
p
r
o
b
s
.
c
p
u
(
)
.
n
u
m
p
y
(
)
)

 
 
 
 
 
 
 
 
v
a
l
_
l
a
b
e
l
s
.
e
x
t
e
n
d
(
y
.
n
u
m
p
y
(
)
)


y
_
v
a
l
_
p
r
o
b
_
p
r
i
t
h
v
i
 
=
 
n
p
.
a
r
r
a
y
(
v
a
l
_
p
r
o
b
s
)

y
_
v
a
l
_
t
r
u
e
_
p
r
i
t
h
v
i
 
=
 
n
p
.
a
r
r
a
y
(
v
a
l
_
l
a
b
e
l
s
)


p
r
e
c
_
v
,
 
r
e
c
_
v
,
 
t
h
r
_
v
 
=
 
p
r
e
c
i
s
i
o
n
_
r
e
c
a
l
l
_
c
u
r
v
e
(
y
_
v
a
l
_
t
r
u
e
_
p
r
i
t
h
v
i
,
 
y
_
v
a
l
_
p
r
o
b
_
p
r
i
t
h
v
i
)

f
1
_
v
 
=
 
2
 
*
 
p
r
e
c
_
v
[
:
-
1
]
 
*
 
r
e
c
_
v
[
:
-
1
]
 
/
 
(
p
r
e
c
_
v
[
:
-
1
]
 
+
 
r
e
c
_
v
[
:
-
1
]
 
+
 
1
e
-
8
)

O
P
T
_
T
H
R
_
P
R
I
T
H
V
I
 
=
 
f
l
o
a
t
(
t
h
r
_
v
[
n
p
.
a
r
g
m
a
x
(
f
1
_
v
)
]
)

p
r
i
n
t
(
f
'
P
r
i
t
h
v
i
 
o
p
t
i
m
a
l
 
t
h
r
e
s
h
o
l
d
 
(
v
a
l
)
:
 
{
O
P
T
_
T
H
R
_
P
R
I
T
H
V
I
:
.
4
f
}
'
)


#
 
T
e
s
t
 
i
n
f
e
r
e
n
c
e

p
r
i
t
h
v
i
_
t
e
s
t
_
e
v
a
l
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(

 
 
 
 
E
m
b
e
d
d
i
n
g
C
l
a
s
s
D
a
t
a
s
e
t
(
P
_
t
e
s
t
_
e
m
b
,
 
y
_
t
e
s
t
)
,

 
 
 
 
b
a
t
c
h
_
s
i
z
e
=
2
5
6
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
0
,

)

t
e
s
t
_
p
r
o
b
s
,
 
t
e
s
t
_
l
a
b
e
l
s
 
=
 
[
]
,
 
[
]

w
i
t
h
 
t
o
r
c
h
.
n
o
_
g
r
a
d
(
)
:

 
 
 
 
f
o
r
 
x
,
 
y
 
i
n
 
p
r
i
t
h
v
i
_
t
e
s
t
_
e
v
a
l
_
l
o
a
d
e
r
:

 
 
 
 
 
 
 
 
l
o
g
i
t
s
 
=
 
b
e
s
t
_
p
r
i
t
h
v
i
(
x
.
t
o
(
D
E
V
I
C
E
)
)

 
 
 
 
 
 
 
 
p
r
o
b
s
 
 
=
 
t
o
r
c
h
.
s
o
f
t
m
a
x
(
l
o
g
i
t
s
,
 
d
i
m
=
1
)
[
:
,
 
1
]

 
 
 
 
 
 
 
 
t
e
s
t
_
p
r
o
b
s
.
e
x
t
e
n
d
(
p
r
o
b
s
.
c
p
u
(
)
.
n
u
m
p
y
(
)
)

 
 
 
 
 
 
 
 
t
e
s
t
_
l
a
b
e
l
s
.
e
x
t
e
n
d
(
y
.
n
u
m
p
y
(
)
)


y
_
p
r
o
b
_
p
r
i
t
h
v
i
 
=
 
n
p
.
a
r
r
a
y
(
t
e
s
t
_
p
r
o
b
s
)

y
_
p
r
e
d
_
p
r
i
t
h
v
i
 
=
 
(
y
_
p
r
o
b
_
p
r
i
t
h
v
i
 
>
=
 
O
P
T
_
T
H
R
_
P
R
I
T
H
V
I
)
.
a
s
t
y
p
e
(
i
n
t
)


p
r
_
a
u
c
_
p
r
i
t
h
v
i
 
=
 
a
v
e
r
a
g
e
_
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
(
y
_
t
e
s
t
,
 
y
_
p
r
o
b
_
p
r
i
t
h
v
i
)

a
u
c
_
p
r
i
t
h
v
i
 
 
 
 
=
 
r
o
c
_
a
u
c
_
s
c
o
r
e
(
y
_
t
e
s
t
,
 
y
_
p
r
o
b
_
p
r
i
t
h
v
i
)

f
1
_
p
r
i
t
h
v
i
 
 
 
 
 
=
 
f
1
_
s
c
o
r
e
(
y
_
t
e
s
t
,
 
y
_
p
r
e
d
_
p
r
i
t
h
v
i
)

p
r
e
c
_
p
r
i
t
h
v
i
 
 
 
=
 
p
r
e
c
i
s
i
o
n
_
s
c
o
r
e
(
y
_
t
e
s
t
,
 
y
_
p
r
e
d
_
p
r
i
t
h
v
i
,
 
z
e
r
o
_
d
i
v
i
s
i
o
n
=
0
)

r
e
c
_
p
r
i
t
h
v
i
 
 
 
 
=
 
r
e
c
a
l
l
_
s
c
o
r
e
(
y
_
t
e
s
t
,
 
y
_
p
r
e
d
_
p
r
i
t
h
v
i
)

a
c
c
_
p
r
i
t
h
v
i
 
 
 
 
=
 
a
c
c
u
r
a
c
y
_
s
c
o
r
e
(
y
_
t
e
s
t
,
 
y
_
p
r
e
d
_
p
r
i
t
h
v
i
)


p
r
i
n
t
(
'
=
=
=
 
P
r
i
t
h
v
i
-
3
0
0
M
 
L
i
n
e
a
r
 
P
r
o
b
e
 
=
=
=
'
)

p
r
i
n
t
(
f
'
P
R
-
A
U
C
 
 
:
 
{
p
r
_
a
u
c
_
p
r
i
t
h
v
i
:
.
4
f
}
'
)

p
r
i
n
t
(
f
'
R
O
C
-
A
U
C
 
:
 
{
a
u
c
_
p
r
i
t
h
v
i
:
.
4
f
}
'
)

p
r
i
n
t
(
f
'
F
1
 
@
 
t
h
r
=
{
O
P
T
_
T
H
R
_
P
R
I
T
H
V
I
:
.
3
f
}
 
:
 
{
f
1
_
p
r
i
t
h
v
i
:
.
4
f
}
'
)

p
r
i
n
t
(
f
'
P
r
e
c
i
s
i
o
n
:
 
{
p
r
e
c
_
p
r
i
t
h
v
i
:
.
4
f
}
 
 
R
e
c
a
l
l
:
 
{
r
e
c
_
p
r
i
t
h
v
i
:
.
4
f
}
 
 
A
c
c
u
r
a
c
y
:
 
{
a
c
c
_
p
r
i
t
h
v
i
:
.
4
f
}
'
)

p
r
i
n
t
(
)

p
r
i
n
t
(
c
l
a
s
s
i
f
i
c
a
t
i
o
n
_
r
e
p
o
r
t
(
y
_
t
e
s
t
,
 
y
_
p
r
e
d
_
p
r
i
t
h
v
i
,
 
t
a
r
g
e
t
_
n
a
m
e
s
=
[
'
N
o
 
C
o
v
e
r
a
g
e
'
,
 
'
H
a
s
 
C
o
v
e
r
a
g
e
'
]
)
)


f
i
g
,
 
a
x
e
s
 
=
 
p
l
t
.
s
u
b
p
l
o
t
s
(
1
,
 
2
,
 
f
i
g
s
i
z
e
=
(
1
2
,
 
5
)
)


c
m
 
=
 
c
o
n
f
u
s
i
o
n
_
m
a
t
r
i
x
(
y
_
t
e
s
t
,
 
y
_
p
r
e
d
_
p
r
i
t
h
v
i
)

i
m
 
=
 
a
x
e
s
[
0
]
.
i
m
s
h
o
w
(
c
m
,
 
i
n
t
e
r
p
o
l
a
t
i
o
n
=
'
n
e
a
r
e
s
t
'
,
 
c
m
a
p
=
'
B
l
u
e
s
'
)

f
i
g
.
c
o
l
o
r
b
a
r
(
i
m
,
 
a
x
=
a
x
e
s
[
0
]
)

a
x
e
s
[
0
]
.
s
e
t
_
x
t
i
c
k
s
(
[
0
,
 
1
]
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
x
t
i
c
k
l
a
b
e
l
s
(
[
'
N
o
 
C
o
v
.
'
,
 
'
H
a
s
 
C
o
v
.
'
]
)

a
x
e
s
[
0
]
.
s
e
t
_
y
t
i
c
k
s
(
[
0
,
 
1
]
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
y
t
i
c
k
l
a
b
e
l
s
(
[
'
N
o
 
C
o
v
.
'
,
 
'
H
a
s
 
C
o
v
.
'
]
)

f
o
r
 
r
 
i
n
 
r
a
n
g
e
(
2
)
:

 
 
 
 
f
o
r
 
c
 
i
n
 
r
a
n
g
e
(
2
)
:

 
 
 
 
 
 
 
 
a
x
e
s
[
0
]
.
t
e
x
t
(
c
,
 
r
,
 
s
t
r
(
c
m
[
r
,
 
c
]
)
,
 
h
a
=
'
c
e
n
t
e
r
'
,
 
v
a
=
'
c
e
n
t
e
r
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
o
l
o
r
=
'
w
h
i
t
e
'
 
i
f
 
c
m
[
r
,
 
c
]
 
>
 
c
m
.
m
a
x
(
)
 
/
 
2
 
e
l
s
e
 
'
b
l
a
c
k
'
,
 
f
o
n
t
s
i
z
e
=
1
4
)

a
x
e
s
[
0
]
.
s
e
t
_
x
l
a
b
e
l
(
'
P
r
e
d
i
c
t
e
d
'
)
;
 
a
x
e
s
[
0
]
.
s
e
t
_
y
l
a
b
e
l
(
'
T
r
u
e
'
)

a
x
e
s
[
0
]
.
s
e
t
_
t
i
t
l
e
(
'
C
o
n
f
u
s
i
o
n
 
M
a
t
r
i
x
'
)


p
r
e
c
_
t
,
 
r
e
c
_
t
,
 
_
 
=
 
p
r
e
c
i
s
i
o
n
_
r
e
c
a
l
l
_
c
u
r
v
e
(
y
_
t
e
s
t
,
 
y
_
p
r
o
b
_
p
r
i
t
h
v
i
)

a
x
e
s
[
1
]
.
p
l
o
t
(
r
e
c
_
t
,
 
p
r
e
c
_
t
,
 
l
w
=
2
,
 
l
a
b
e
l
=
f
'
P
R
-
A
U
C
 
=
 
{
p
r
_
a
u
c
_
p
r
i
t
h
v
i
:
.
3
f
}
'
)

a
x
e
s
[
1
]
.
a
x
h
l
i
n
e
(
y
_
t
e
s
t
.
m
e
a
n
(
)
,
 
l
i
n
e
s
t
y
l
e
=
'
-
-
'
,
 
c
o
l
o
r
=
'
g
r
e
y
'
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
B
a
s
e
l
i
n
e
 
(
{
y
_
t
e
s
t
.
m
e
a
n
(
)
:
.
3
f
}
)
'
)

a
x
e
s
[
1
]
.
s
c
a
t
t
e
r
(
[
r
e
c
_
p
r
i
t
h
v
i
]
,
 
[
p
r
e
c
_
p
r
i
t
h
v
i
]
,
 
m
a
r
k
e
r
=
'
*
'
,
 
s
=
2
0
0
,
 
c
o
l
o
r
=
'
r
e
d
'
,
 
z
o
r
d
e
r
=
5
,

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
l
a
b
e
l
=
f
'
V
a
l
-
o
p
t
 
t
h
r
=
{
O
P
T
_
T
H
R
_
P
R
I
T
H
V
I
:
.
3
f
}
'
)

a
x
e
s
[
1
]
.
s
e
t
_
x
l
a
b
e
l
(
'
R
e
c
a
l
l
'
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
a
b
e
l
(
'
P
r
e
c
i
s
i
o
n
'
)

a
x
e
s
[
1
]
.
s
e
t
_
t
i
t
l
e
(
'
P
r
e
c
i
s
i
o
n
-
R
e
c
a
l
l
 
C
u
r
v
e
'
)

a
x
e
s
[
1
]
.
l
e
g
e
n
d
(
l
o
c
=
'
u
p
p
e
r
 
r
i
g
h
t
'
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
x
l
i
m
(
[
0
,
 
1
]
)
;
 
a
x
e
s
[
1
]
.
s
e
t
_
y
l
i
m
(
[
0
,
 
1
]
)


p
l
t
.
s
u
p
t
i
t
l
e
(
'
P
r
i
t
h
v
i
-
3
0
0
M
 
L
i
n
e
a
r
 
P
r
o
b
e
'
,
 
f
o
n
t
s
i
z
e
=
1
3
,
 
f
o
n
t
w
e
i
g
h
t
=
'
b
o
l
d
'
)

p
l
t
.
t
i
g
h
t
_
l
a
y
o
u
t
(
)

p
l
t
.
s
a
v
e
f
i
g
(
F
I
G
U
R
E
S
_
D
I
R
 
/
 
'
n
b
1
9
_
p
r
i
t
h
v
i
_
l
i
n
e
a
r
_
e
v
a
l
.
p
n
g
'
,
 
d
p
i
=
1
5
0
,
 
b
b
o
x
_
i
n
c
h
e
s
=
'
t
i
g
h
t
'
)

p
l
t
.
s
h
o
w
(
)


d
e
l
 
b
e
s
t
_
p
r
i
t
h
v
i

t
o
r
c
h
.
c
u
d
a
.
e
m
p
t
y
_
c
a
c
h
e
(
)

## Step 7: Cross-Model Comparison

In [ ]:
comparison_rows = [
    {'Model': 'XGBoost (Engineered)',   'PR-AUC': pr_auc_xgb,     'ROC-AUC': auc_xgb,
     'F1': f1_xgb,     'Precision': prec_xgb,     'Recall': rec_xgb,     'Accuracy': acc_xgb,
     'Threshold': OPT_THR_XGB},
    {'Model': 'MOSAIKS + Ridge',        'PR-AUC': pr_auc_mosaiks,  'ROC-AUC': auc_mosaiks,
     'F1': f1_mosaiks,  'Precision': prec_mosaiks,  'Recall': rec_mosaiks,  'Accuracy': acc_mosaiks,
     'Threshold': OPT_THR_MOSAIKS},
    {'Model': 'ResNet-18 + LogReg',     'PR-AUC': pr_auc_resnet,  'ROC-AUC': auc_resnet,
     'F1': f1_resnet,  'Precision': prec_resnet,  'Recall': rec_resnet,  'Accuracy': acc_resnet,
     'Threshold': OPT_THR_RESNET},
    {'Model': 'DINOv2 + LogReg',        'PR-AUC': pr_auc_dino,    'ROC-AUC': auc_dino,
     'F1': f1_dino,    'Precision': prec_dino,    'Recall': rec_dino,    'Accuracy': acc_dino,
     'Threshold': OPT_THR_DINO},
    {'Model': 'Prithvi-300M Linear',    'PR-AUC': pr_auc_prithvi, 'ROC-AUC': auc_prithvi,
     'F1': f1_prithvi, 'Precision': prec_prithvi, 'Recall': rec_prithvi, 'Accuracy': acc_prithvi,
     'Threshold': OPT_THR_PRITHVI},
]

comparison = pd.DataFrame(comparison_rows).sort_values('PR-AUC', ascending=False).reset_index(drop=True)

print('=== Phase 1 Binary Classification — NE India Test Set ===')
print(comparison.to_string(index=False))

comparison.to_csv(RESULTS_DIR / 'nb19_combined_binary_comparison.csv', index=False)
print(f'\nSaved: {RESULTS_DIR / "nb19_combined_binary_comparison.csv"}')

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(14, 5))

labels = comparison['Model'].tolist()
x = np.arange(len(labels))
w = 0.25

ax.bar(x - w, comparison['PR-AUC'],  w, label='PR-AUC',  color='#1f77b4')
ax.bar(x,     comparison['ROC-AUC'], w, label='ROC-AUC', color='#ff7f0e')
ax.bar(x + w, comparison['F1'],      w, label='F1',      color='#2ca02c')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Phase 1 Binary Classification — All Models\nNE India test set')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Overlaid PR curves for all models
fig, ax = plt.subplots(figsize=(9, 7))

models_pr = [
    ('XGBoost (Engineered)',  y_prob_xgb,             pr_auc_xgb),
    ('MOSAIKS + Ridge',      y_scores_test_mosaiks,   pr_auc_mosaiks),
    ('ResNet-18 + LogReg',   y_prob_resnet,           pr_auc_resnet),
    ('DINOv2 + LogReg',      y_prob_dino,             pr_auc_dino),
    ('Prithvi-300M Linear',  y_prob_prithvi,          pr_auc_prithvi),
]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for (name, scores, prauc), color in zip(models_pr, colors):
    prec_c, rec_c, _ = precision_recall_curve(y_test, scores)
    ax.plot(rec_c, prec_c, lw=2, color=color,
            label=f'{name} (PR-AUC={prauc:.3f})')

ax.axhline(y_test.mean(), linestyle='--', color='grey', alpha=0.6,
           label=f'Random baseline ({y_test.mean():.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Phase 1 Models\nNE India test set', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_all_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Save Models & Push to GitHub

In [ ]:
import joblib

joblib.dump(xgb_model, MODELS_DIR / 'nb19_xgboost_engineered.pkl')
joblib.dump(ridge,     MODELS_DIR / 'nb19_mosaiks_ridge.pkl')
joblib.dump(lr_resnet, MODELS_DIR / 'nb19_resnet18_linear.pkl')
joblib.dump(lr_dino,   MODELS_DIR / 'nb19_dinov2_linear.pkl')
joblib.dump(scaler_r,  MODELS_DIR / 'nb19_resnet18_scaler.pkl')
joblib.dump(scaler_d,  MODELS_DIR / 'nb19_dinov2_scaler.pkl')

with open(MODELS_DIR / 'nb19_prithvi_linear_ckpt.txt', 'w') as f:
    f.write(checkpoint_cb.best_model_path)

# Per-model metrics CSVs
for row in comparison_rows:
    safe = row['Model'].lower().replace(' ', '_').replace('-', '').replace('(', '').replace(')', '').replace('+', '')
    pd.DataFrame([row]).to_csv(RESULTS_DIR / f'nb19_{safe}_metrics.csv', index=False)

print('All models and metrics saved.')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

# Auto-discover output files
files = ['notebooks/02_binary_baselines.ipynb']
for pat in ['nb19_*']:
    files.extend([str(p) for p in FIGURES_DIR.glob(pat)])
    files.extend([str(p) for p in RESULTS_DIR.glob(pat)])
    files.extend([str(p) for p in MODELS_DIR.glob(pat)])

for f in files:
    if os.path.exists(f):
        subprocess.run(['git', 'add', '-f', f], check=True)
    else:
        print(f'Skipping (not yet generated): {f}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB02: Binary classification baselines (XGBoost + MOSAIKS + ResNet-18 + DINOv2 + Prithvi)'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')